# Arrow interoperability - Rust

All 9 Rust examples from [docs/arrow.md](https://platob.github.io/yggdryl/arrow/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and expect the
[evcxr](https://github.com/evcxr/evcxr) kernel. Declare the crate once, in
a cell of your own, before running them:

```rust
:dep yggdryl = { version = "0.1", features = ["parquet", "iceberg"] }
```

In [ ]:
use yggdryl::arrow::DefaultArrowScalar;
use yggdryl::{DataType, Field};

let field = Field::new("symbol", DataType::Utf8, false);
let scalar = field.default_arrow_scalar()?;

// A scalar is one Arrow row plus the exact Field that owns it.
assert_eq!(scalar.field(), &field);
assert_eq!(scalar.data_type(), &DataType::Utf8);
assert_eq!(scalar.array().len(), 1);
assert_eq!(scalar.to_value()?.as_str(), Some(""));

## Nullability picks the default

In [ ]:
use yggdryl::arrow::DefaultArrowScalar;
use yggdryl::{DataType, Field, Value};

// A bare DataType has no name of its own, so it borrows a required one.
let scalar = DataType::Int64.default_arrow_scalar()?;
assert_eq!(scalar.field().name(), "value");
assert!(!scalar.field().is_nullable());
assert_eq!(scalar.to_value()?.as_i128(), Some(0));

// A nullable Field defaults to a logical null and keeps its own identity.
let optional = Field::new("symbol", DataType::Utf8, true).default_arrow_scalar()?;
assert!(optional.array().is_null(0));
assert_eq!(optional.to_value()?, Value::Null);

// A required Null column has no value it could ever hold.
let refused = Field::new("never", DataType::Null, false).default_arrow_scalar();
assert!(refused.is_err());

## A struct root is one row

In [ ]:
use yggdryl::arrow::DefaultArrowScalar;
use yggdryl::{DataType, Field, Value};

let schema = Field::new(
    "row",
    DataType::from_fields([
        DataType::Int64.required_field("id"),
        DataType::Utf8.nullable_field("symbol"),
    ])?,
    false,
);

let row = schema.default_arrow_scalar()?.to_value()?;
assert_eq!(row.len(), 2);
assert_eq!(row.get(0).and_then(Value::as_i128), Some(0));
assert_eq!(row.get(1), Some(&Value::Null));

## ArrowScalar

In [ ]:
use std::sync::Arc;

use arrow_array::{ArrayRef, Int64Array};
use yggdryl::arrow::ArrowScalar;
use yggdryl::{DataType, Field, Value};

let field = Field::new("id", DataType::Int64, false);
let scalar = ArrowScalar::from_value(field.clone(), Value::from(7_i64))?;
assert_eq!(scalar.data_type(), &DataType::Int64);
assert_eq!(scalar.to_value()?.as_i128(), Some(7));

// The parts come apart and go back together unchanged.
let (field, array) = scalar.into_parts();
let rebuilt = ArrowScalar::from_parts(field, array)?;
assert_eq!(rebuilt.into_value()?.as_i128(), Some(7));

// A foreign array has to be one row of the Field's exact physical datatype.
let two: ArrayRef = Arc::new(Int64Array::from(vec![1, 2]));
assert!(ArrowScalar::from_parts(Field::new("id", DataType::Int64, false), two).is_err());

// And the value has to satisfy the Field, recursively.
assert!(ArrowScalar::from_value(Field::new("id", DataType::Int64, false), Value::Null).is_err());

## StructScalar

In [ ]:
use std::sync::Arc;

use arrow_array::{ArrayRef, Datum, Int64Array, StringArray, StructArray};
use yggdryl::arrow::{StructScalar, schema_from_field};
use yggdryl::{DataType, Field};

let schema = Field::new(
    "row",
    DataType::from_fields([
        DataType::Int64.required_field("id"),
        DataType::Utf8.nullable_field("symbol"),
    ])?,
    false,
);

let projected = schema.to_arrow_schema()?;
let array = StructArray::new(
    projected.fields().clone(),
    vec![
        Arc::new(Int64Array::from(vec![1])) as ArrayRef,
        Arc::new(StringArray::from(vec![Some("AAPL")])) as ArrayRef,
    ],
    None,
);

let row = StructScalar::from_parts(schema, array)?;
assert_eq!(row.field().name(), "row");
assert_eq!(row.schema().field_len(), 2);

// Children come back as zero-copy one-element slices, by position or by name.
let (field, column) = row.entry(0).expect("first column");
assert_eq!(field.name(), "id");
assert_eq!(column.len(), 1);
assert_eq!(row.get_by_name("symbol").map(|column| column.len()), Some(1));

// Arrow's own scalar marker is a shallow clone away.
let marker = row.to_arrow_scalar();
let (inner, is_scalar) = marker.get();
assert!(is_scalar);
assert_eq!(inner.len(), 1);

## Projecting a schema

In [ ]:
use yggdryl::arrow::{record_schema_from_arrow, record_schema_to_arrow, schema_from_field};
use yggdryl::{DataType, Field};

let schema = Field::from_parts(
    "row",
    DataType::from_fields([
        DataType::Int64.required_field("id"),
        DataType::Utf8.nullable_field("symbol"),
    ])?,
    false,
    [("owner", "trading")],
)?;

// Root metadata becomes Arrow schema metadata, and comes back.
let projected = record_schema_to_arrow(&schema)?;
assert_eq!(projected.fields().len(), 2);
assert_eq!(projected.metadata().get("owner").map(String::as_str), Some("trading"));
assert_eq!(record_schema_from_arrow("row", &projected)?, schema);

// schema_from_field is the same projection, already behind an Arc.
assert_eq!(schema.to_arrow_schema()?.as_ref(), &projected);

// A root that is not a non-null Struct is refused, not coerced.
assert!(record_schema_to_arrow(&Field::new("row", DataType::Int64, false)).is_err());
assert!(record_schema_to_arrow(&schema.with_nullable(true)).is_err());

## Streaming batches

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch, RecordBatchReader, StringArray};
use yggdryl::io::Buffer;
use yggdryl::ipc::{self, IpcOptions};
use yggdryl::{DataType, Field};

let schema = Field::new(
    "row",
    DataType::from_fields([
        DataType::Int64.required_field("id"),
        DataType::Utf8.nullable_field("symbol"),
    ])?,
    false,
);
let projected = schema.to_arrow_schema()?;

let batch = |ids: Vec<i64>, symbols: Vec<Option<&str>>| {
    RecordBatch::try_new(
        Arc::clone(&projected),
        vec![
            Arc::new(Int64Array::from(ids)),
            Arc::new(StringArray::from(symbols)),
        ],
    )
};

let mut handle = Buffer::new();
let options = IpcOptions::new();
ipc::write_batch_reader(
    &mut handle,
    yggdryl::arrow::batch_reader(
        Arc::clone(&projected),
        [batch(vec![1, 2], vec![Some("AAPL"), None])?, batch(vec![3], vec![None])?],
    ),
    &options,
)?;

// A BatchReader knows its schema before it yields anything.
let reader = ipc::read_batch_reader(&handle, None, &options)?;
assert_eq!(reader.schema().as_ref(), projected.as_ref());

let mut rows = 0;
for batch in reader {
    rows += batch?.num_rows();
}
assert_eq!(rows, 3);

## Materialization budgets

In [ ]:
use yggdryl::arrow::ArrowScalar;
use yggdryl::{DataType, Field, Value};

// One logical null, one million and one mandatory physical child slots.
let items = Field::new(
    "items",
    DataType::fixed_size_list(Field::new("item", DataType::Int32, false), 1_000_001)?,
    true,
);
let message = ArrowScalar::from_value(items, Value::Null).unwrap_err().to_string();
assert!(message.contains("expanded slots"), "{message}");
assert!(message.contains("expected at most 1000000"), "{message}");
assert!(message.contains("got 1000001"), "{message}");

// Fixed width is counted across siblings, not per column.
let wide = DataType::from_fields([
    Field::new("left", DataType::fixed_size_binary(40 * 1024 * 1024)?, false),
    Field::new("right", DataType::fixed_size_binary(40 * 1024 * 1024)?, false),
])?;
let message = ArrowScalar::from_value(Field::new("wide", wide, true), Value::Null)
    .unwrap_err()
    .to_string();
assert!(message.contains("fixed bytes"), "{message}");
assert!(message.contains("expected at most 67108864"), "{message}");

In [ ]:
use yggdryl::arrow::ArrowScalar;
use yggdryl::{DataType, Field, UnionMode, Value};

// The inactive branch is far past the byte budget, and is never visited.
let dense = DataType::union(
    [
        (0, Field::new("selected", DataType::Int32, false)),
        (
            1,
            Field::new(
                "inactive",
                DataType::fixed_size_binary(64 * 1024 * 1024 + 1)?,
                false,
            ),
        ),
    ],
    UnionMode::Dense,
)?;

let chosen = Value::from_sequence([Value::from(0_i8), Value::from(11_i32)]);
let scalar = ArrowScalar::from_value(Field::new("choice", dense, false), chosen.clone())?;
assert_eq!(scalar.to_value()?, chosen);